In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnableBranch, RunnableSequence
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables.graph_ascii import draw_ascii
from typing import Literal
from pydantic import BaseModel
from dotenv import find_dotenv, load_dotenv
import os

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path)
apiKey = os.getenv("OLLAMA_API_KEY")
os.environ['OLLAMA_API_KEY'] = apiKey
llm = ChatOllama(model="gpt-oss:120b-cloud")

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
# Example 1 - Sequential Chains with pipes
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain = prompt_1 | llm | parser | prompt_2 | llm | parser
response = chain.invoke({"country": "France"})

print(response)


Here are five of the most famous tourist attractions you’ll find in **Paris**, the capital city of France:

| # | Tourist Place | Why It’s Famous | Quick Tips |
|---|----------------|----------------|------------|
| 1 | **Eiffel Tower** | The iconic iron lattice tower that defines the Paris skyline; offers spectacular views from its 2nd and 3rd‑level observation decks. | Buy tickets online in advance; sunrise or sunset visits are especially magical. |
| 2 | **Louvre Museum** | The world’s largest art museum, home to masterpieces such as the *Mona Lisa* and the *Venus de Milo*. | Allocate at least 3‑4 hours; consider a guided audio tour to navigate the vast collections. |
| 3 | **Cathédrale Notre‑Dam de Paris** | A masterpiece of French Gothic architecture (still under restoration, but its façade and surrounding Île de la Cité remain a must‑see). | Visit the crypt and the nearby Sainte‑Chapelle for stunning stained‑glass windows. |
| 4 | **Montmartre & Sacré‑Cœur Basilica** | A historic

In [4]:
# Example 2 - Sequential Chains with RunnableSequence
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="Name 5 famous tourist places of {city}"
)
parser = StrOutputParser()
chain_1 = prompt_1 | llm | parser
chain_2 = prompt_2 | llm | parser
sequence = RunnableSequence(chain_1, chain_2)
response = sequence.invoke({"country": "China"})

print(response)


Sure! Here are five of the most famous tourist attractions in Beijing, China:

| # | Tourist Spot | Why It’s Famous |
|---|--------------|----------------|
| 1️⃣ | **The Forbidden Palace (Imperial Palace) / Palace Museum** | The sprawling former imperial residence of Ming and Qing dynasties; a UNESCO World Heritage site with over 9,000 rooms, exquisite palaces, courtyards, and priceless artifacts. |
| 2️⃣ | **The Great Wall (Mutianyu, Badaling, Jinshanling sections)** | One of the world’s most iconic ancient fortifications; the Beijing‑adjacent sections offer spectacular mountain scenery and well‑preserved watchtowers. |
| 3️⃣ | **Tiananmen Square** | The world’s largest city square, flanked by the Monument to the People’s Heroes, the Great Hall of the People, the National Museum, and the iconic portrait of Mao Zedong. |
| 4️⃣ | **Temple of Heaven (Tiantan Park)** | A beautifully designed Ming‑Dynasty temple complex where emperors prayed for good harvests; famous for its striking Hall 

In [2]:
# Example 3 - Parallel Chains
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Which is the capital city of {country}"
)

prompt_2 = PromptTemplate(
    input_variables=["city"],
    template="In which continent {country} is located?"
)
parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {
        "response1": prompt_1 | llm | parser,
        "response2": prompt_2 | llm | parser
    }
)
response = parallel_chain.invoke({"country": "India"})
print(response)


{'response1': 'The capital city of India is **New\u202fDelhi**.', 'response2': 'India is located on the continent of **Asia** (specifically in the South\u202fAsia region).'}


In [4]:
# Example 4 - Parallel + Sequencial Chains
prompt_1 = PromptTemplate(
    input_variables=["country"],
    template="Write the independence history of {country} in 3 lines"
)

prompt_2 = PromptTemplate(
    input_variables=["country"],
    template="Write notes on tourist places in {country} in 3 lines"
)
parser = StrOutputParser()

parallel_chain = RunnableParallel(
    {
        "history": prompt_1 | llm | parser,
        "notes": prompt_2 | llm | parser
    }
)

prompt_3 = PromptTemplate(
    input_variables=["history", "notes"],
    template="Write summarizing paragraph of a country having history as {history} and tourist places as {notes} in 5 lines"
)
parser = StrOutputParser()
summarizing_chain = prompt_3 | llm | parser

final_chain = parallel_chain | summarizing_chain

response = final_chain.invoke({"country": "USA"})
print(response)


The United States was born from colonial grievances over taxation and lack of representation, sparking the Revolutionary War in 1775 and the Declaration of Independence on July 4, 1776.  
Victories at Saratoga (1777) and the decisive siege of Yorktown (1781), supported by France, secured American freedom, cemented by the Treaty of Paris on September 3, 1783.  
Today, New York City’s iconic skyline—Statue of Liberty, Times Square, Empire State Building—paired with world‑class museums like MoMA and the MET, makes it a cultural powerhouse.  
From the breathtaking vistas of the Grand Canyon to San Francisco’s Golden Gate Bridge and Yellowstone’s geysers and wildlife, the nation boasts unparalleled natural marvels.  
New Orleans’ rich French, Spanish, and African heritage shines in its jazz, Creole cuisine, historic French Quarter, and the year‑round celebration of Mardi Gras.


In [6]:
# Example 5 - Conditional chains
parser = StrOutputParser()

# Structured output schema for sentiment
class FeedbackSentiment(BaseModel):
    sentiment: Literal["positive", "negative"]

sentiment_parser =  PydanticOutputParser(pydantic_object=FeedbackSentiment)
sentiment_prompt = PromptTemplate(
    template="Analyze the sentiment of the following customer feedback and classify it as positive or negative.\n\nFeedback: {feedback} and provide feedback in following format: {response_format}",
    input_variables=["feedback"],
    partial_variables={"response_format": sentiment_parser.get_format_instructions()}
)

sentiment_chain = sentiment_prompt | llm | sentiment_parser

# Positive reply chain
positive_prompt = PromptTemplate.from_template(
    "Write a warm, genuine thank-you response to this positive customer feedback:\n\n{feedback}"
)
positive_chain = positive_prompt | llm | parser

# Negative reply chain
negative_prompt = PromptTemplate.from_template(
    "Write a sincere, empathetic apology and offer assistance for this negative customer feedback:\n\n{feedback}"
)
negative_chain = negative_prompt | llm | parser

# Default chain (fallback)
default_chain = RunnableLambda(lambda: "Thank you for reaching out. We will get back to you shortly.")

# Conditional branching
branch = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == "positive", positive_chain),
    (lambda x: x["sentiment"].sentiment == "negative", negative_chain),
    default_chain
)

# Full pipeline
def run_pipeline(feedback_text):
    sentiment_result = sentiment_chain.invoke({"feedback": feedback_text})
    reply = branch.invoke({"sentiment": sentiment_result, "feedback": feedback_text})
    return reply

# Test it
#print(run_pipeline("I absolutely loved your product! It changed my life."))
print(run_pipeline("The service was terrible and nothing worked as expected. Hotel room was not very clean."))





Dear [Guest Name],

Thank you for taking the time to share your experience with us. I’m truly sorry to hear that your recent stay fell far short of the standards we aim to uphold.  

It is disappointing to learn that the service you received was “terrible,” that the amenities did not work as expected, and that your room was not as clean as it should have been. These issues are unacceptable, and I understand how frustrating and unsettling they must have been, especially when you were looking forward to a comfortable, hassle‑free stay.

Please know that we take your feedback very seriously. I have already alerted our Front‑Desk manager, Housekeeping Supervisor, and Maintenance Team so they can investigate what went wrong and ensure that similar lapses do not happen again. Your comments are driving immediate corrective actions, including additional training for staff and a thorough review of our cleaning and inspection procedures.

In the meantime, I would like to make things right for yo